# Part 3 - Genişletilmiş Veri Seti ile Qwen 3.5 4B Testi (Batch Inference)

Bu notebook, ikinci aşamada Claude API ile 100 sorudan 2500 soruya çıkarılan (augmented) matematik kelime problemleri veri setinin **Qwen/Qwen3.5-4B** modeli ile test edilmesini sağlar.

Modelin 2500 soruyu makul bir sürede çözebilmesi için **Batch Inference (Toplu Çıkarım)** yöntemi kullanılmıştır. Notebook ayrıca uzun süren işlemler için bir "kaldığı yerden devam etme" (checkpoint) mekanizması ve strateji bazlı detaylı doğruluk (accuracy) analizleri içerir.

## 1. Kurulum ve Ortam Hazırlığı
Gerekli kütüphaneleri kuruyor ve Google Drive'ımızı bağlayarak veri setimize erişim sağlıyoruz.

In [ ]:
!pip install -q transformers accelerate torch

from google.colab import drive
drive.mount('/content/drive')

## 2. Model ve Tokenizer Yükleme
`Qwen/Qwen3.5-4B` modelini bellek optimizasyonu sağlamak adına `float16` veri tipinde yüklüyoruz. Toplu çıkarım (batch inference) yapacağımız için tokenizer'ın `padding_side` ayarını `'left'` olarak belirliyoruz.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
import re
import time
import os

MODEL_ID = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'  # Decoder-only modeller için batch işleminde zorunludur

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Model yüklendi.")

## 3. Yardımcı Fonksiyonlar
Modelin ürettiği metinlerden sayısal sonuçları çekmek ve doğruluğunu kontrol etmek için fonksiyonlarımızı tanımlıyoruz. `generate_answers_batch` fonksiyonu, GPU'yu tam kapasite kullanarak aynı anda birden fazla soruyu çözer.

In [ ]:
def extract_number(text):
    """Metinden son sayıyı regex ile çıkarır ve temizler."""
    text = str(text).replace(',', '').replace('.', '')
    numbers = re.findall(r'-?\d+(?:\.\d+)?', text)
    if numbers:
        try:
            return float(numbers[-1])
        except:
            return None
    return None

def is_correct(model_output, expected_answer):
    """Modelin tahmini ile beklenen cevabı karşılaştırır."""
    try:
        predicted = extract_number(model_output)
        expected = float(str(expected_answer).replace(',', '').replace('.', ''))
        if predicted is None:
            return False
        return abs(predicted - expected) < 0.01
    except:
        return False

def generate_answers_batch(questions):
    """Soruları batch (grup) olarak modele gönderir ve çözer."""
    prompts = [
        f"""Aşağıdaki matematik problemini adım adım çöz ve sonunda sadece sayısal cevabı yaz.

Problem: {q}

Çözüm:""" for q in questions
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    responses = []
    for output in outputs:
        new_tokens = output[inputs['input_ids'].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)
        responses.append(response.strip())
    return responses

print("Fonksiyonlar tanımlandı.")

## 4. Veri Yükleme ve Checkpoint (Kaldığı Yerden Devam) Kontrolü
2500 sorunun tamamlanması uzun sürebileceğinden, notebook çökerse veya durdurulursa işlemlerin baştan başlamaması için mevcut durumu kontrol ediyoruz.

In [ ]:
RESULTS_PATH = "/content/drive/MyDrive/kaynaklar/augmented_dataset_with_results.csv"
SOURCE_PATH  = "/content/drive/MyDrive/kaynaklar/augmented_dataset_final_2500.csv"
BATCH_SIZE   = 16

if os.path.exists(RESULTS_PATH):
    df_aug = pd.read_csv(RESULTS_PATH)
    islenen = int(df_aug['qwen_correct'].notna().sum())
    print(f"Kaldığı yerden devam: {islenen} soru işlenmiş.")
else:
    df_aug = pd.read_csv(SOURCE_PATH)
    df_aug['qwen_output']  = None
    df_aug['qwen_correct'] = None
    islenen = 0
    print("İlk kez başlatılıyor...")

kalan_df = df_aug[df_aug['qwen_correct'].isna()].copy()
print(f"Toplam: {len(df_aug)} | İşlenen: {islenen} | Kalan: {len(kalan_df)}")

## 5. Hata Kurtarma (Recovery)
Önceki çalıştırmalarda Out of Memory (OOM) veya başka hatalar sebebiyle "HATA" olarak işaretlenmiş satırlar varsa, bunları yeniden işlenmek üzere sıfırlıyoruz.

In [ ]:
df_aug = pd.read_csv(RESULTS_PATH)
hata_mask = df_aug['qwen_output'] == 'HATA'

df_aug.loc[hata_mask, 'qwen_output']  = None
df_aug.loc[hata_mask, 'qwen_correct'] = None
df_aug.to_csv(RESULTS_PATH, index=False)

print(f"Sıfırlanan soru sayısı: {hata_mask.sum()}")
print(f"Kalan işlenecek: {df_aug['qwen_correct'].isna().sum()}")

## 6. Ana Test Döngüsü
Kalan verileri `BATCH_SIZE` grupları halinde modele besliyoruz. Olası bir bellek taşmasında (Memory Error) işlemi iptal etmek yerine `try-except` bloğu ile soruları tek tek çözmeyi deniyoruz.

In [ ]:
toplam = len(kalan_df)
islenen_bu_oturum = 0
baslangic = time.time()

for batch_start in range(0, toplam, BATCH_SIZE):
    batch    = kalan_df.iloc[batch_start:batch_start + BATCH_SIZE]
    questions = batch['generated_question'].tolist()
    expected  = batch['expected_number'].tolist()
    indices   = batch.index.tolist()

    try:
        responses = generate_answers_batch(questions)
        for idx, response, exp in zip(indices, responses, expected):
            df_aug.at[idx, 'qwen_output']  = response
            df_aug.at[idx, 'qwen_correct'] = is_correct(response, exp)

    except Exception as e:
        print(f"Batch hatası ({batch_start}): {e}")
        # Hata durumunda batch içindekileri tekil (fallback) olarak dene
        for idx, q, exp in zip(indices, questions, expected):
            try:
                resp = generate_answers_batch([q])[0]
                df_aug.at[idx, 'qwen_output']  = resp
                df_aug.at[idx, 'qwen_correct'] = is_correct(resp, exp)
            except:
                df_aug.at[idx, 'qwen_output']  = "HATA"
                df_aug.at[idx, 'qwen_correct'] = False

    islenen_bu_oturum += len(batch)

    # Her 100 soruda bir kaydet ve istatistikleri yazdır
    if islenen_bu_oturum % 100 == 0:
        df_aug.to_csv(RESULTS_PATH, index=False)
        gecen      = (time.time() - baslangic) / 60
        kalan_sure = (gecen / islenen_bu_oturum) * (toplam - islenen_bu_oturum)
        islenentoplam = int(df_aug['qwen_correct'].notna().sum())
        dogru         = int(df_aug['qwen_correct'].sum())

        print(f"[{islenentoplam}/2500] Doğruluk: %{dogru/islenentoplam*100:.1f} | "
              f"Geçen: {gecen:.1f}dk | Kalan tahmini: {kalan_sure:.1f}dk | Yedek alındı.")

# Döngü bittikten sonra son durumu kaydet
df_aug.to_csv(RESULTS_PATH, index=False)
print(f"\nTamamlandı! Toplam doğruluk: %{df_aug['qwen_correct'].mean()*100:.1f}")

## 7. Strateji Bazında Sonuçların Analizi
Test bittikten sonra, modelin hangi stratejilerde daha çok zorlandığını ve orijinalde başardığı/başaramadığı sorular üzerindeki performans değişimini analiz ediyoruz.

In [ ]:
df_aug = pd.read_csv(RESULTS_PATH)

print("\n--- Genel Doğruluk ---")
print(f"Toplam: {len(df_aug)} | Doğru: {int(df_aug['qwen_correct'].sum())} | "
      f"Oran: %{df_aug['qwen_correct'].mean()*100:.1f}")

print("\n--- Strateji Bazında Doğruluk ---")
ozet = df_aug.groupby(['strategy', 'original_status'])['qwen_correct'].agg(['sum', 'count', 'mean'])
ozet['mean'] = (ozet['mean'] * 100).round(1)
ozet.columns = ['Doğru', 'Toplam', 'Oran (%)']
print(ozet.to_string())

## 8. Hata Kontrolü ve Manuel Test
Son olarak "HATA" almış veya boş (`None`) dönmüş veri var mı diye kontrol ediyor ve pipeline'ın düzgün çalıştığından emin olmak için ufak bir manuel test atıyoruz.

In [ ]:
df_aug = pd.read_csv(RESULTS_PATH)
print("HATA sayısı:", (df_aug['qwen_output'] == 'HATA').sum())
print("None sayısı:", df_aug['qwen_output'].isna().sum())
print("\nEn Sık Tekrar Eden Yanıtlar (Örneklem):")
print(df_aug['qwen_output'].value_counts().head(10))

print("\n--- Tek Bir Batch Manuel Test ---")
test_questions = df_aug['generated_question'].iloc[0:2].tolist()
print("Sorular:", test_questions)

try:
    result = generate_answers_batch(test_questions)
    print("Sonuç:", result)
except Exception as e:
    print("Hata:", e)
    import traceback
    traceback.print_exc()